# 11. cal01_base_v6 만들기

| 항목 | 내용 |
| --- | --- |
| 입력 | `cal01_base_v1.csv` (원기록), 검토 표 `data/labels/domain_decisions.csv` |
| 사용하는 검토 | 검토 표의 VERSION이 v2, v3, v6인 행 (v6_보류, v6_제안은 쓰지 않음) |
| 출력 | `data/processed/cal01_base_v6.csv` (1분 묶음) |
| 이 버전에서 하는 일 | v5에 팀 판단(검토 사이트, 팀 판단 정리)을 더함 |

**v6 팀 판단** (26건)

| DOMAIN_LABEL | 번호 | 내용 |
| --- | --- | --- |
| 1 | T01 ~ T13, T26, T28, T31 | 팀원이 이상으로 본 구간 (CP 개별 이탈, Z3 온도 진동과 출력 포화, 공통 움직임과 다른 방향의 CP 등) |
| 0 | T14 ~ T19, T25, T27, T29, T30 | 팀원이 정상으로 본 구간 (작은 변화, 다른 LOT에도 있는 모양, 정상 제어 등) |

- 1차 공통 움직임 해제(0)와 부딪히는 판단(T20 ~ T22)과 기준이 정해지지 않은 판단(T23, T24)은 v6_보류로 두고 쓰지 않음

**라벨 규칙**

| 단계 | 1 | 0 | 빈칸 |
| --- | --- | --- | --- |
| 원기록 `구역__DOMAIN_LABEL` (08과 같음) | 구역 센서 중 하나라도 확정 구간에 있음 | STAT 1이고, \|Z\| > 3인 구역 센서가 모두 해제됨 | 그 밖 |
| 1분 `구역__DOMAIN_LABEL` (10과 같음) | 1분 안에 DOMAIN 1이 하나라도 있음 | STAT 1인 원기록이 모두 해제됨 | 그 밖, 제외한 분 |

- 같은 구간에 해제(0)와 확정(1)이 겹치면 1을 씀

**v1 ~ v6 한눈에 보기** (버전마다 한 가지씩만 더함)

| 버전 | 단위 | STAT_LABEL | DOMAIN_LABEL | 만드는 노트북 |
| --- | --- | --- | --- | --- |
| v1 | 원기록 (1초) | 같은 LOT 기준 \|Z\| > 3 | 없음 | 06 |
| v2 | 원기록 (1초) | v1과 같음 | 공통 움직임 검토로 해제한 곳 0 | 07 |
| v3 | 원기록 (1초) | v1과 같음 | v2 + 새로 찾은 공통 움직임 해제 0 | 08 |
| v4 | 1분 | 1분 안에 후보가 하나라도 있으면 1 | v2를 1분으로 묶음 | 09 |
| v5 | 1분 | v4와 같음 | v3를 1분으로 묶음 | 10 |
| v6 | 1분 | v4와 같음 | v3 + 팀 판단(1, 0)을 1분으로 묶음 | 11 |

## 1. 공통 경로

In [ ]:
import os
import pandas as pd

# 저장소 루트 찾기: notebooks 폴더가 안 보이면 한 칸 위로 (최대 두 번)
ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "notebooks")):
        ROOT = os.path.dirname(ROOT)

# 폴더
DATA_DIR = os.path.join(ROOT, "data")                    # 원본 CSV 8개
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")             # data/raw에 두었으면 그 폴더를 씀
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")  # 만든 CSV (cal01_base_v1 ~ v6)
LABEL_DIR = os.path.join(ROOT, "data", "labels")         # 팀 검토 표 (domain_decisions.csv)
FIG_DIR = os.path.join(ROOT, "figures")                  # 그림
ENC = "utf-8-sig"

# 원본 CSV
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")

# 읽을 가공 CSV (이름만 바꾸면 됨)
DATA_FILE = "cal01_base_v1.csv"  # v1을 읽음
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)

# 만들 CSV
OUTPUT_FILE = "cal01_base_v6.csv"
OUT_CSV = os.path.join(PROCESSED_DIR, OUTPUT_FILE)

# 검토 표와, 그중 사용할 VERSION
DECISION_CSV = os.path.join(LABEL_DIR, "domain_decisions.csv")
USE_VERSIONS = ["v2", "v3", "v6"]

# 마지막에 비교할 v5
COMPARE_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v5.csv")

import numpy as np

## 2. 구역과 센서

In [ ]:
# 구역별 센서 (CP-Z4M은 값이 0.0000000001 수준이라 라벨 계산에 쓰지 않음)
GROUPS = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}

# 라벨 계산에 쓰는 센서 21개
SENSORS = []
for zone in GROUPS:
    SENSORS = SENSORS + GROUPS[zone]

# CSV에 저장하는 원본 센서 22개 (CP-Z4M은 저장만 하고 계산에는 안 씀)
SAVE_SENSORS = SENSORS + ["CP-Z4M"]

# 라벨 컬럼 22개 (구역마다 STAT, DOMAIN)
LABEL_COLUMNS = []
for zone in GROUPS:
    LABEL_COLUMNS.append(zone + "__STAT_LABEL")
    LABEL_COLUMNS.append(zone + "__DOMAIN_LABEL")

## 3. v1과 검토 표 읽기

In [ ]:
raw = pd.read_csv(BASE_CSV, encoding=ENC, low_memory=False)  # low_memory=False: 컬럼 형식을 파일 전체를 보고 정함
raw["MEAS_DT"] = pd.to_datetime(raw["MEAS_DT"], errors="raise")

decisions = pd.read_csv(DECISION_CSV, encoding=ENC)
decisions["START"] = pd.to_datetime(decisions["START"], errors="raise")
decisions["END"] = pd.to_datetime(decisions["END"], errors="raise")

# 이 버전에서 쓸 검토만 남김
decisions = decisions[decisions["VERSION"].isin(USE_VERSIONS)].reset_index(drop=True)

print("원기록:", raw.shape)
decisions.groupby(["VERSION", "DOMAIN_LABEL"]).size()

## 4. 센서별 Z-score (v1과 같은 계산)

In [ ]:
# 제외 행을 뺀 데이터로 LOT별 평균, 표준편차를 구함
use = raw[raw["EXCLUDED"] == False]
lot_mean = use.groupby("LOT_NO")[SENSORS].mean()
lot_std = use.groupby("LOT_NO")[SENSORS].std(ddof=0)

# 센서마다 |Z| = |값 - LOT 평균| / LOT 표준편차
z = pd.DataFrame(np.nan, index=raw.index, columns=SENSORS)

for sensor in SENSORS:
    mean = raw["LOT_NO"].map(lot_mean[sensor])
    std = raw["LOT_NO"].map(lot_std[sensor])
    std = std.replace(0, np.nan)

    z[sensor] = (raw[sensor] - mean).abs() / std

# 제외 행은 Z를 비워둠
z.loc[raw["EXCLUDED"] == True] = np.nan

## 5. 검토 구간 표시하기

센서마다 두 가지 표를 만듦.

| 표 | True인 곳 |
| --- | --- |
| `cleared` | 검토해서 후보를 해제한 행 (DOMAIN 0) |
| `confirmed` | 검토해서 이상으로 확정한 행 (DOMAIN 1) |

In [ ]:
cleared = pd.DataFrame(False, index=raw.index, columns=SENSORS)
confirmed = pd.DataFrame(False, index=raw.index, columns=SENSORS)

for i in range(len(decisions)):
    row = decisions.loc[i]

    same_lot = raw["LOT_NO"] == row["LOT_NO"]
    in_time = (raw["MEAS_DT"] >= row["START"]) & (raw["MEAS_DT"] <= row["END"])
    hit = same_lot & in_time

    if row["DOMAIN_LABEL"] == 0:
        cleared.loc[hit, row["SENSOR"]] = True
    else:
        confirmed.loc[hit, row["SENSOR"]] = True

print("해제 표시가 있는 행:", int(cleared.any(axis=1).sum()))
print("확정 표시가 있는 행:", int(confirmed.any(axis=1).sum()))

## 6. 원기록 DOMAIN_LABEL 붙이기 (08과 같은 규칙)

| 상황 | DOMAIN_LABEL |
| --- | --- |
| 구역 센서 중 하나라도 확정 구간에 있음 | 1 |
| STAT 1이고, \|Z\| > 3인 구역 센서가 모두 해제됨 | 0 |
| 그 밖 | 빈칸 |

In [ ]:
for zone in GROUPS:
    zone_sensors = GROUPS[zone]
    stat_col = zone + "__STAT_LABEL"
    domain_col = zone + "__DOMAIN_LABEL"

    # |Z| > 3인데 아직 해제되지 않은 센서가 남아 있는지
    over = z[zone_sensors] > 3
    still_open = (over & (cleared[zone_sensors] == False)).any(axis=1)

    is_candidate = raw[stat_col] == 1
    is_known = raw[stat_col].notna()
    is_confirmed = confirmed[zone_sensors].any(axis=1)

    raw[domain_col] = np.nan
    raw.loc[is_candidate & (still_open == False), domain_col] = 0
    raw.loc[is_known & is_confirmed, domain_col] = 1

## 7. 1분씩 묶기 (10과 같은 규칙)

| 컬럼 | 1분 값 |
| --- | --- |
| 센서 (예: `TC-Z4`) | 1분 평균 |
| `센서__MIN`, `센서__MAX`, `센서__STD` | 1분 안의 최소, 최대, 표준편차 |
| `ROW_COUNT` | 1분 안의 원기록 행 수 |
| `EXCLUDED` | 1분 안에 제외 행이 하나라도 있으면 True |

원기록 행마다 도메인 표시를 먼저 만듦.

| 표시 | True인 경우 |
| --- | --- |
| `구역_D1` | DOMAIN_LABEL이 1 |
| `구역_OPEN` | STAT_LABEL이 1인데 DOMAIN_LABEL이 빈칸 (아직 해제되지 않은 후보) |

In [ ]:
# 초를 버리고 분까지만 남김 (예: 10:50:37 -> 10:50:00)
raw["MINUTE"] = raw["MEAS_DT"].dt.floor("min")

for zone in GROUPS:
    stat = raw[zone + "__STAT_LABEL"]
    domain = raw[zone + "__DOMAIN_LABEL"]

    raw[zone + "_D1"] = domain == 1
    raw[zone + "_OPEN"] = (stat == 1) & domain.isna()

grouped = raw.groupby(["LOT_NO", "MINUTE"])

base = grouped["GRADE"].first().reset_index()
base = base.rename(columns={"MINUTE": "MEAS_DT"})

base["ROW_COUNT"] = grouped["MEAS_DT"].count().values
base["EXCLUDED"] = grouped["EXCLUDED"].max().values
base["EXCLUSION_REASON"] = grouped["EXCLUSION_REASON"].first().values

for sensor in SAVE_SENSORS:
    base[sensor] = grouped[sensor].mean().values
    base[sensor + "__MIN"] = grouped[sensor].min().values
    base[sensor + "__MAX"] = grouped[sensor].max().values
    base[sensor + "__STD"] = grouped[sensor].std().values

# 컬럼을 하나씩 많이 붙였으므로 표를 한 번 정리함 (속도 경고 방지)
base = base.copy()

print("1분 묶음:", base.shape)

## 8. 1분 라벨 붙이기

| 컬럼 | 1 | 0 | 빈칸 |
| --- | --- | --- | --- |
| `구역__STAT_LABEL` | 1분 안에 원기록 STAT 1이 하나라도 있음 | 모두 0 | 제외한 분, 판단할 행이 없음 |
| `구역__DOMAIN_LABEL` | 1분 안에 DOMAIN 1이 하나라도 있음 | STAT 1인 원기록이 모두 해제됨 | 그 밖 |

In [ ]:
for zone in GROUPS:
    stat_col = zone + "__STAT_LABEL"
    domain_col = zone + "__DOMAIN_LABEL"

    has_one = grouped[zone + "_D1"].max().values
    has_open = grouped[zone + "_OPEN"].max().values

    # 1분 안에 STAT 1이 하나라도 있으면 1
    base[stat_col] = grouped[stat_col].max().values

    base[domain_col] = np.nan
    base.loc[(base[stat_col] == 1) & (has_open == False), domain_col] = 0
    base.loc[has_one == True, domain_col] = 1

# 제외한 분은 라벨을 비워둠
for column in LABEL_COLUMNS:
    base.loc[base["EXCLUDED"] == True, column] = np.nan

base = base.copy()

## 9. 저장 (컬럼은 v5와 같음)

| 컬럼 묶음 | 컬럼 |
| --- | --- |
| 식별 | `ROW_ID`, `LOT_NO`, `MEAS_DT` (1분 시작 시각), `GRADE`, `ROW_COUNT` |
| 제외 | `EXCLUDED`, `EXCLUSION_REASON` |
| 센서 22개 x 4 | 센서(평균), `__MIN`, `__MAX`, `__STD` |
| 라벨 22개 | `구역__STAT_LABEL`, `구역__DOMAIN_LABEL` |

In [ ]:
# 행 이름: LOT_분 (예: 240001_20240102T0336)
base["ROW_ID"] = base["LOT_NO"].astype(str) + "_" + base["MEAS_DT"].dt.strftime("%Y%m%dT%H%M")

# 라벨을 1.0 대신 1로 저장하려고 정수형(Int64)으로 바꿈. Int64는 빈칸도 담을 수 있음
for column in LABEL_COLUMNS:
    base[column] = base[column].astype("Int64")

sensor_columns = []
for sensor in SAVE_SENSORS:
    sensor_columns = sensor_columns + [sensor, sensor + "__MIN", sensor + "__MAX", sensor + "__STD"]

columns = ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE", "ROW_COUNT", "EXCLUDED", "EXCLUSION_REASON"] + sensor_columns + LABEL_COLUMNS
base = base[columns]

base.to_csv(OUT_CSV, index=False, encoding=ENC)
print("저장:", OUTPUT_FILE, base.shape)

## 10. 구역별 라벨 개수

In [ ]:
# 구역별 개수
rows = []

for zone in GROUPS:
    stat = base[zone + "__STAT_LABEL"]
    domain = base[zone + "__DOMAIN_LABEL"]

    rows.append({
        "구역": zone,
        "STAT 1": int((stat == 1).sum()),
        "STAT 0": int((stat == 0).sum()),
        "STAT 빈칸": int(stat.isna().sum()),
        "DOMAIN 1": int((domain == 1).sum()),
        "DOMAIN 0": int((domain == 0).sum()),
    })

zone_counts = pd.DataFrame(rows)
zone_counts

## 11. v5와 비교

모델 노트북과 같은 정답 `LABEL = DOMAIN_LABEL` (빈칸이면 `STAT_LABEL`)로 비교함. 같은 1분(`ROW_ID`)끼리 맞춤.

| 컬럼 | 뜻 |
| --- | --- |
| `0 → 1` | v5에서 0이었는데 v6에서 1이 된 1분 |
| `1 → 0` | v5에서 1이었는데 v6에서 0이 된 1분 |

In [ ]:
v5 = pd.read_csv(COMPARE_CSV, encoding=ENC, low_memory=False)

# 같은 1분끼리 붙이기
both = base[["ROW_ID", "LOT_NO"]].merge(v5[["ROW_ID"]], on="ROW_ID", how="inner")
both = both.set_index("ROW_ID")
v6_by_row = base.set_index("ROW_ID")
v5_by_row = v5.set_index("ROW_ID")

label5 = pd.DataFrame(index=both.index)
label6 = pd.DataFrame(index=both.index)

for zone in GROUPS:
    label5[zone] = v5_by_row.loc[both.index, zone + "__DOMAIN_LABEL"].fillna(v5_by_row.loc[both.index, zone + "__STAT_LABEL"])
    label6[zone] = v6_by_row.loc[both.index, zone + "__DOMAIN_LABEL"].fillna(v6_by_row.loc[both.index, zone + "__STAT_LABEL"])

rows = []

for zone in GROUPS:
    rows.append({
        "구역": zone,
        "v5 LABEL 1": int((label5[zone] == 1).sum()),
        "v6 LABEL 1": int((label6[zone] == 1).sum()),
        "0 → 1": int(((label5[zone] == 0) & (label6[zone] == 1)).sum()),
        "1 → 0": int(((label5[zone] == 1) & (label6[zone] == 0)).sum()),
    })

print("v5:", len(v5), "/ v6:", len(base), "/ 같은 1분:", len(both))
compare = pd.DataFrame(rows)
compare

## 12. 팀 판단별로 바뀐 1분

In [ ]:
# 센서가 속한 구역
zone_of = {}
for zone in GROUPS:
    for sensor in GROUPS[zone]:
        zone_of[sensor] = zone

team = decisions[decisions["VERSION"] == "v6"].reset_index(drop=True)
minute = base.set_index("ROW_ID").loc[both.index]

rows = []

for i in range(len(team)):
    row = team.loc[i]
    zone = zone_of[row["SENSOR"]]

    same_lot = minute["LOT_NO"] == row["LOT_NO"]
    in_time = (minute["MEAS_DT"] >= row["START"].floor("min")) & (minute["MEAS_DT"] <= row["END"])
    window = same_lot & in_time

    before = label5.loc[window, zone]
    after = label6.loc[window, zone]

    rows.append({
        "번호": row["DECISION_ID"],
        "LOT": row["LOT_NO"],
        "센서": row["SENSOR"],
        "판단": row["DOMAIN_LABEL"],
        "1분": int(window.sum()),
        "제외한 분": int(minute.loc[window, "EXCLUDED"].sum()),
        "0 → 1": int(((before == 0) & (after == 1)).sum()),
        "1 → 0": int(((before == 1) & (after == 0)).sum()),
    })

by_decision = pd.DataFrame(rows)
by_decision